In [1]:
!pip -q install qiskit qiskit-aer numpy



from qiskit import QuantumCircuit, transpile

from qiskit.quantum_info import Statevector

from qiskit_aer import AerSimulator

import numpy as np

import random

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.8/9.8 MB 28.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 47.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 17.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.9/54.9 kB 1.9 MB/s eta 0:00:00


In [2]:
from qiskit import QuantumCircuit, transpile

from qiskit_aer import AerSimulator



def build_superposition_circuit():

    circuit = QuantumCircuit(1, 1)

    circuit.h(0)

    circuit.measure(0, 0)

    return circuit



def run_trial(backend, circuit, shot_count):

    compiled_circuit = transpile(circuit, backend)

    execution = backend.run(compiled_circuit, shots=shot_count)

    return execution.result().get_counts()



backend = AerSimulator()

trial_sizes = [100, 1000, 10000]



for n_shots in trial_sizes:

    circuit = build_superposition_circuit()

    outcome_counts = run_trial(backend, circuit, n_shots)

    zero_prob = outcome_counts.get('0', 0) / n_shots

    one_prob = outcome_counts.get('1', 0) / n_shots

    print(f"--- Shots = {n_shots} ---")

    print(f"Raw counts: {outcome_counts}")

    print(f"P(0) = {zero_prob:.4f}, P(1) = {one_prob:.4f}\n")

--- Shots = 100 ---
Raw counts: {'0': 41, '1': 59}
P(0) = 0.4100, P(1) = 0.5900

--- Shots = 1000 ---
Raw counts: {'0': 477, '1': 523}
P(0) = 0.4770, P(1) = 0.5230

--- Shots = 10000 ---
Raw counts: {'1': 5065, '0': 4935}
P(0) = 0.4935, P(1) = 0.5065



In [3]:
from qiskit import QuantumCircuit, transpile

from qiskit_aer import AerSimulator



def prepare_plus_state():

    qc = QuantumCircuit(1, 1)

    qc.h(0)

    return qc



def prepare_minus_state():

    qc = QuantumCircuit(1, 1)

    qc.x(0)

    qc.h(0)

    return qc



def measure_in_x_basis(qc):

    qc.h(0)

    qc.measure(0, 0)

    return qc



backend = AerSimulator()

n_shots = 1024



for label_, prep_fn in [("|+>", prepare_plus_state), ("|->", prepare_minus_state)]:

    circuit = measure_in_x_basis(prep_fn())

    compiled = transpile(circuit, backend)

    counts = backend.run(compiled, shots=n_shots).result().get_counts()

    print(f"X-basis measurement of {label_} state: {counts}")

X-basis measurement of |+> state: {'0': 1024}
X-basis measurement of |-> state: {'1': 1024}


In [4]:
from qiskit import QuantumCircuit

from qiskit.quantum_info import Statevector



def make_bell_pair():

    qc = QuantumCircuit(2)

    qc.h(0)

    qc.cx(0, 1)

    return qc



bell_circuit = make_bell_pair()

sv = Statevector.from_instruction(bell_circuit)



print("Entangled statevector:")

print(sv)

print("\nOutcome probabilities:")

probs = sv.probabilities_dict()

for outcome, p in probs.items():

    print(f"  |{outcome}>: {p:.4f}")

Entangled statevector:
Statevector([0.70710678+0.j, 0.        +0.j, 0.        +0.j,
             0.70710678+0.j],
            dims=(2, 2))

Outcome probabilities:
  |00>: 0.5000
  |11>: 0.5000


In [5]:
from qiskit import QuantumCircuit, transpile

from qiskit_aer import AerSimulator

import numpy as np



def build_rotated_circuit(angle):

    qc = QuantumCircuit(1, 1)

    qc.ry(angle, 0)

    qc.measure(0, 0)

    return qc



def estimate_angle_from_p0(p0):

    p0_clamped = min(max(p0, 0.0), 1.0)

    return 2 * np.arccos(np.sqrt(p0_clamped))



true_angle = np.pi / 3   # unknown angle to be recovered

n_shots = 5000



backend = AerSimulator()

circuit = build_rotated_circuit(true_angle)

compiled = transpile(circuit, backend)

counts = backend.run(compiled, shots=n_shots).result().get_counts()



p0 = counts.get('0', 0) / n_shots

p1 = counts.get('1', 0) / n_shots

recovered_angle = estimate_angle_from_p0(p0)



print(f"Measurement counts: {counts}")

print(f"P(0) = {p0:.4f}, P(1) = {p1:.4f}")

print(f"True theta (rad): {true_angle:.4f}")

print(f"Estimated theta (rad): {recovered_angle:.4f}")

print(f"Estimated theta (deg): {np.degrees(recovered_angle):.4f}")

Measurement counts: {'1': 1244, '0': 3756}
P(0) = 0.7512, P(1) = 0.2488
True theta (rad): 1.0472
Estimated theta (rad): 1.0444
Estimated theta (deg): 59.8411


In [6]:
from qiskit import QuantumCircuit, transpile

from qiskit_aer import AerSimulator

import random



def quantum_random_bits(backend, n_shots):

    qc = QuantumCircuit(1, 1)

    qc.h(0)

    qc.measure(0, 0)

    compiled = transpile(qc, backend)

    return backend.run(compiled, shots=n_shots).result().get_counts()



def biased_classical_bits(n_shots, bias_toward_zero=0.65, seed=7):

    rng = random.Random(seed)

    tally = {"0": 0, "1": 0}

    for _ in range(n_shots):

        bit = "0" if rng.random() < bias_toward_zero else "1"

        tally[bit] += 1

    return tally



n_shots = 1000

backend = AerSimulator()



q_counts = quantum_random_bits(backend, n_shots)

c_counts = biased_classical_bits(n_shots)



print(f"Quantum source counts: {q_counts}")

print(f"Classical (biased) source counts: {c_counts}\n")

print("Quantum probabilities:")

print(f"  P(0) = {q_counts.get('0',0)/n_shots:.3f}, P(1) = {q_counts.get('1',0)/n_shots:.3f}")

print("Classical probabilities:")

print(f"  P(0) = {c_counts['0']/n_shots:.3f}, P(1) = {c_counts['1']/n_shots:.3f}")

Quantum source counts: {'1': 504, '0': 496}
Classical (biased) source counts: {'0': 671, '1': 329}

Quantum probabilities:
  P(0) = 0.496, P(1) = 0.504
Classical probabilities:
  P(0) = 0.671, P(1) = 0.329
